In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from data_extractor import DataExtractor


sns.set_theme()

In [ ]:
db_connection = os.environ.get("MYSQL_CON_STRING")
DAYS_FOR_REPORT = 1

In [ ]:
data_extractor = DataExtractor(db_connection, DAYS_FOR_REPORT)

In [ ]:
feed_df = data_extractor.get_feed_df()
article_df = data_extractor.get_article_df(feed_df)
article_topics_df = data_extractor.get_article_topics_df(feed_df)
new_article_topics_df = data_extractor.get_article_new_topics_df(feed_df)
language_df = data_extractor.get_language_df()
bookmark_df = data_extractor.get_bookmark_df()
data_extractor.add_stats_to_feed(feed_df, article_df)
user_reading_time_df = data_extractor.get_user_reading_activity(
    language_df, feed_df
)
user_exercise_time_df = data_extractor.get_user_exercise_activity()
combined_user_activity_df = (
    data_extractor.get_combined_user_reading_exercise_activity(
        user_exercise_time_df, user_reading_time_df
    )
)
topic_reading_time_df = data_extractor.get_topic_reading_time()
total_unique_articles_opened_by_users = len(
    article_df[article_df.id.isin(user_reading_time_df.id)]
)
exercise_activity_df = data_extractor.get_exercise_type_activity()

In [ ]:
article_topics_df.Topic.value_counts()

In [ ]:
new_article_topics_df.Topic.shape, article_topics_df.Topic.shape

In [ ]:
new_article_topics_df.Topic.value_counts()

In [ ]:
new_article_topics_df.origin_type.value_counts()

In [ ]:
user_reading_time_df = data_extractor.get_user_reading_activity(
    language_df, feed_df
)

In [ ]:
len(article_df[article_df.id.isin(user_reading_time_df.id)])

In [ ]:
user_reading_time_df

In [ ]:
user_reading_time_df.Language.value_counts().reset_index().sort_values("count")

In [ ]:
article_df

In [ ]:
top_5_articles_by_opened = user_reading_time_df.groupby(["Language", "Feed Name"]).id.value_counts().reset_index().sort_values("count", ascending=False)[:5]

In [ ]:
top_5_articles_by_opened.merge(article_df[["id","title"]], on="id")[["Language", "Feed Name", "id", "title", "count"]]

In [ ]:
from nltk.tokenize import sent_tokenize
from collections import Counter
from tqdm import tqdm

def normalize_sent(text: str):
    return text.lower().strip()

def sent_count(text: str):
    return Counter([normalize_sent(sent) for paragraph in text.split("\n\n") for sent in sent_tokenize(paragraph) if len(sent.strip()) > 10 ])

def get_total_sent_counts(text_list: list[str]):
    total_counts = Counter()
    for text in tqdm(text_list, total=len(text_list)):
        total_counts += sent_count(text)
    return total_counts

In [ ]:
total_sent_count = get_total_sent_counts(last_week_articles.content)

In [ ]:
total_sent_count

In [ ]:
new_sents = total_sent_count - previous_counter
new_sents

In [ ]:
import os
import json

In [ ]:
data_loaded = {}
for file in os.listdir("report_data"):
    with open(os.path.join("report_data", file), "r", encoding="utf-8") as f:
        data_loaded[file] = json.load(f)

In [ ]:
data_loaded.keys()

In [ ]:
data_loaded['da-crawl-19_06_24_06_56_30.json']["feeds"]

In [ ]:
from collections import Counter

def accumulate_date_for_language(lang, loaded_data):
    accumulated_dict = {}
    accumulated_dict["feeds"] = {}
    for key in loaded_data.keys():
        current_dict = loaded_data[key]
        if lang in key:
            print(f"Reading... '{key}'")
            accumulated_dict["total_time"] = accumulated_dict.get("total_time", []) + [current_dict["total_time"]]
            accumulated_dict["date"] = accumulated_dict.get("date", []) + [key.split("-")[-1]]
            for feed in current_dict["feeds"].keys():
                if feed not in accumulated_dict["feeds"]:
                    accumulated_dict["feeds"][feed] = {}
                for key, val in current_dict["feeds"][feed].items():
                    if key == "article_data":
                        if key not in accumulated_dict["feeds"][feed]:
                            accumulated_dict["feeds"][feed][key] = {}
                        for art_key, art_val in current_dict["feeds"][feed]["article_data"].items():
                            accumulated_dict["feeds"][feed][key][art_key] = Counter(accumulated_dict["feeds"][feed].get(key, {}).get(art_key, {})) + Counter(art_val)
                    else:
                        accumulated_dict["feeds"][feed][key] = accumulated_dict["feeds"][feed].get(key, []) + [val]
    return accumulated_dict


In [ ]:
data_loaded['da-crawl-19_06_24_06_56_30.json'].keys()

In [ ]:
acc_dict = accumulate_date_for_language("", data_loaded)

In [ ]:
acc_dict["feeds"].keys()

In [ ]:
acc_dict["feeds"]

In [ ]:
def get_total_quality_counts(acc_dict):
    total_counts = Counter()
    for feed in acc_dict["feeds"]:
        if "article_data" not in acc_dict["feeds"][feed]:
            continue
        if "quality_error" not in acc_dict["feeds"][feed]["article_data"]:
            continue
        total_counts += Counter(acc_dict["feeds"][feed]["article_data"]["quality_error"])
    return total_counts

In [ ]:
total_rejected_article_reasons = dict(get_total_quality_counts(acc_dict))

In [ ]:
total_rejected_article_reasons
total_rejected_article_reasons["Total"] = sum(total_rejected_article_reasons.values())

In [ ]:
pd_quality_errors = pd.DataFrame.from_dict(total_rejected_article_reasons, orient="index").reset_index()
pd_quality_errors.columns = ["Reason", "Count"]

In [ ]:
print(pd_quality_errors.to_html(index=False))

In [ ]:
previous_counter = total_sent_count